# Notebook 4: Quantum Algorithms

This notebook is part of the FDP hands-on session on Quantum Algorithms. We cover:
1. **Deutsch-Jozsa Algorithm**: Determining if an $n$-bit oracle is constant or balanced in one query.
2. **Bernstein-Vazirani Algorithm**: Reconstructing a hidden bitstring $a \in \{0,1\}^n$ in one query.
3. **Simon's Algorithm**: Finding a hidden period $s \in \{0,1\}^n$ by measuring orthogonal vectors.

We implement these algorithms in both **Qiskit** and **PennyLane**.

## Section 1: Deutsch-Jozsa Algorithm (n=3)

The Deutsch-Jozsa algorithm determines if a function $f: \{0,1\}^n \to \{0,1\}$ is constant or balanced in a single query. We implement it for $n=3$ using two oracles:
* **Constant Oracle**: $f(x) = 1$ (flips target qubit for all inputs).
* **Balanced Oracle**: $f(x) = x_0 \oplus x_1 \oplus x_2$ (flips target based on XOR sum of inputs).

In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
import numpy as np

# 1. Define Qiskit Oracles
def get_dj_oracle_qiskit(oracle_type):
    qc = QuantumCircuit(4) # 3 controls (0, 1, 2), 1 target (3)
    if oracle_type == "constant":
        qc.x(3) # Always flip target
    elif oracle_type == "balanced":
        qc.cx(0, 3) # XOR of all inputs
        qc.cx(1, 3)
        qc.cx(2, 3)
    return qc

# 2. Construct Deutsch-Jozsa Circuit
def run_dj_qiskit(oracle_type):
    qc = QuantumCircuit(4, 3) # 4 qubits, 3 classical bits to measure control register
    
    # Initialize target in |->
    qc.x(3)
    qc.h(3)
    
    # Put controls in superposition
    qc.h([0, 1, 2])
    qc.barrier()
    
    # Apply oracle
    oracle = get_dj_oracle_qiskit(oracle_type)
    qc.compose(oracle, inplace=True)
    qc.barrier()
    
    # Apply final Hadamards on controls
    qc.h([0, 1, 2])
    
    # Measure controls
    qc.measure([0, 1, 2], [0, 1, 2])
    return qc

# Run both simulations
sim = AerSimulator()
for o_type in ["constant", "balanced"]:
    qc_dj = run_dj_qiskit(o_type)
    counts = sim.run(qc_dj, shots=100).result().get_counts()
    outcome = list(counts.keys())[0]
    classification = "Constant" if outcome == '000' else "Balanced"
    print(f"Oracle: {o_type:8} | Measured Q0Q1Q2: {outcome} ({classification})")

Oracle: constant | Measured Q0Q1Q2: 000 (Constant)
Oracle: balanced | Measured Q0Q1Q2: 111 (Balanced)


### Deutsch-Jozsa in PennyLane

In [2]:
import pennylane as qml

dev_dj = qml.device("default.qubit", wires=4)

def apply_dj_oracle_pl(oracle_type):
    if oracle_type == "constant":
        qml.PauliX(wires=3)
    elif oracle_type == "balanced":
        qml.CNOT(wires=[0, 3])
        qml.CNOT(wires=[1, 3])
        qml.CNOT(wires=[2, 3])

@qml.qnode(dev_dj)
def run_dj_pl(oracle_type):
    # Initialize target in |->
    qml.PauliX(wires=3)
    qml.Hadamard(wires=3)
    
    # Controls in |+>
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=1)
    qml.Hadamard(wires=2)
    
    apply_dj_oracle_pl(oracle_type)
    
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=1)
    qml.Hadamard(wires=2)
    
    return qml.probs(wires=[0, 1, 2])

for o_type in ["constant", "balanced"]:
    probs = run_dj_pl(o_type)
    print(f"Oracle: {o_type:8} | P(000): {probs[0]:.2f} | P(others): {(1-probs[0]):.2f}")

Oracle: constant | P(000): 1.00 | P(others): 0.00
Oracle: balanced | P(000): 0.00 | P(others): 1.00


## Section 2: Bernstein-Vazirani Algorithm

The Bernstein-Vazirani algorithm finds a hidden bitstring $a \in \{0,1\}^n$ given an oracle that computes $f(x) = a \cdot x \pmod 2$.

We choose the hidden string $a = 101$. The oracle implements CNOT gates from the control register to the target on bits where $a_i = 1$ (qubits 0 and 2).

In [3]:
# Hidden string a = 101
# Qiskit circuit construction
qc_bv = QuantumCircuit(4, 3)

# Initialize target in |->
qc_bv.x(3)
qc_bv.h(3)

# Initialize controls in superposition
qc_bv.h([0, 1, 2])
qc_bv.barrier()

# Oracle: CNOT from Q0 and Q2 to target Q3 (since hidden a = 101)
qc_bv.cx(0, 3)
qc_bv.cx(2, 3)
qc_bv.barrier()

# Final Hadamards on controls
qc_bv.h([0, 1, 2])

# Measure controls
qc_bv.measure([0, 1, 2], [0, 1, 2])

print("Bernstein-Vazirani Circuit Diagram:")
print(qc_bv.draw(output='text'))

counts_bv = sim.run(qc_bv, shots=10).result().get_counts()
print("\nMeasured Hidden string (Qiskit little-endian format q2 q1 q0):")
print(counts_bv)

Bernstein-Vazirani Circuit Diagram:
     ┌───┐      ░            ░ ┌───┐┌─┐      
q_0: ┤ H ├──────░───■────────░─┤ H ├┤M├──────
     ├───┤      ░   │        ░ ├───┤└╥┘┌─┐   
q_1: ┤ H ├──────░───┼────────░─┤ H ├─╫─┤M├───
     ├───┤      ░   │        ░ ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├──────░───┼────■───░─┤ H ├─╫──╫─┤M├
     ├───┤┌───┐ ░ ┌─┴─┐┌─┴─┐ ░ └───┘ ║  ║ └╥┘
q_3: ┤ X ├┤ H ├─░─┤ X ├┤ X ├─░───────╫──╫──╫─
     └───┘└───┘ ░ └───┘└───┘ ░       ║  ║  ║ 
c: 3/════════════════════════════════╩══╩══╩═
                                     0  1  2 

Measured Hidden string (Qiskit little-endian format q2 q1 q0):
{'101': 10}


## Section 3: Simon's Algorithm

Simon's algorithm solves a promise problem: find a hidden period $s \neq 0^n$ such that $f(x) = f(y) \iff x \oplus y = s$.

We choose $s = 011$ (on $n=3$ qubits). 
An oracle for $s = 011$ is implemented by projecting inputs $x$ to $f(x)$ such that $f(x) = f(x \oplus 011)$:
* Target wire 0 receives copy of $x_0$: `cx(0, 3)`
* Target wire 1 receives $x_1 \oplus x_2$: `cx(1, 4)` and `cx(2, 4)`
* Target wire 2 remains $|0\rangle$.

This gives $f(x) = (x_0, x_1 \oplus x_2, 0)$ which satisfies $f(x) = f(x \oplus 011)$.

In [4]:
# Qiskit Simon's Algorithm Circuit
# 3 control qubits (0, 1, 2), 3 target qubits (3, 4, 5)
qc_simon = QuantumCircuit(6, 3)

# Prepare control superposition
qc_simon.h([0, 1, 2])
qc_simon.barrier()

# Apply Oracle for s = 011
qc_simon.cx(0, 3)
qc_simon.cx(1, 4)
qc_simon.cx(2, 4)
qc_simon.barrier()

# Measure second register (target qubits) to collapse first register
# Note: In simulation, measuring second register is conceptually important 
# but mathematically we can just apply final Hadamards and measure controls directly.
qc_simon.h([0, 1, 2])

# Measure control register
qc_simon.measure([0, 1, 2], [0, 1, 2])

print("Simon's Algorithm Circuit Diagram:")
print(qc_simon.draw(output='text'))

counts_simon = sim.run(qc_simon, shots=1000).result().get_counts()
print("\nMeasured output vectors y (which must satisfy y . s = 0 mod 2):")
print(counts_simon)

# Verify orthogonality: for each output y, y0*s0 + y1*s1 + y2*s2 = 0 mod 2
# Since s = 011 (binary: s0=0, s1=1, s2=1), we check if y1 + y2 is even.
s = np.array([0, 1, 1])
all_orthogonal = True
for outcome in counts_simon.keys():
    # Parse outcome (Qiskit format: index 0 is Q2, index 1 is Q1, index 2 is Q0)
    y = np.array([int(outcome[2]), int(outcome[1]), int(outcome[0])])
    product = np.dot(y, s) % 2
    if product != 0:
        all_orthogonal = False
        print(f"Vector {outcome} is NOT orthogonal to s!")
        
if all_orthogonal:
    print("\nSuccess: All measured outcomes are orthogonal to the hidden period s = 011!")
    print("Measured vectors belong to: {000, 100, 011, 111}.")

Simon's Algorithm Circuit Diagram:
     ┌───┐ ░                 ░ ┌───┐┌─┐      
q_0: ┤ H ├─░───■─────────────░─┤ H ├┤M├──────
     ├───┤ ░   │             ░ ├───┤└╥┘┌─┐   
q_1: ┤ H ├─░───┼────■────────░─┤ H ├─╫─┤M├───
     ├───┤ ░   │    │        ░ ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├─░───┼────┼────■───░─┤ H ├─╫──╫─┤M├
     └───┘ ░ ┌─┴─┐  │    │   ░ └───┘ ║  ║ └╥┘
q_3: ──────░─┤ X ├──┼────┼───░───────╫──╫──╫─
           ░ └───┘┌─┴─┐┌─┴─┐ ░       ║  ║  ║ 
q_4: ──────░──────┤ X ├┤ X ├─░───────╫──╫──╫─
           ░      └───┘└───┘ ░       ║  ║  ║ 
q_5: ──────░─────────────────░───────╫──╫──╫─
           ░                 ░       ║  ║  ║ 
c: 3/════════════════════════════════╩══╩══╩═
                                     0  1  2 

Measured output vectors y (which must satisfy y . s = 0 mod 2):
{'111': 270, '000': 250, '001': 245, '110': 235}

Success: All measured outcomes are orthogonal to the hidden period s = 011!
Measured vectors belong to: {000, 100, 011, 111}.
